# Mini Projects

This notebook collects mini projects that put the calculus helpers into practice: detecting extrema, approximating π, and comparing the numerical methods in one place. Each project ends with suggested exercises so you can extend the exploration.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

assets_dir = root / "assets" / "figures"
assets_dir.mkdir(parents=True, exist_ok=True)

## Max/min detection

We scan a smooth function for sign changes in its numerical derivative, then locally refine the bracket to nail down the extremum. This emulates the idea behind calculus tests for maxima and minima.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from calculus_with_python.differentiation import difference_quotient
from calculus_with_python.examples import sin
from calculus_with_python.grids import make_grid

def f(x):
    return sin(x) + 0.2 * x
x = make_grid(-3.0, 3.0, 1201)
fx = f(x)
deriv = difference_quotient(f, x, h=1e-5, method="central")

sign_changes = np.where(np.sign(deriv[:-1]) * np.sign(deriv[1:]) < 0)[0]
extrema = []
for idx in sign_changes:
    a, b = x[idx], x[idx + 1]
    for _ in range(25):
        mid = 0.5 * (a + b)
        if difference_quotient(f, mid, h=1e-6) == 0:
            break
        if difference_quotient(f, a, h=1e-6) * difference_quotient(f, mid, h=1e-6) < 0:
            b = mid
        else:
            a = mid
    extrema.append((mid, f(mid)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, fx, label="f(x)")
for px, py in extrema:
    ax.scatter(px, py, color="red", zorder=5)
    ax.annotate("extremum", (px, py), textcoords="offset points", xytext=(0, 8), ha="center")
ax.set_title("Max/min detection using derivative sign changes")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.grid(True)
fig.savefig(assets_dir / "mini_project_extrema.png", dpi=150, bbox_inches="tight")
plt.close(fig)
fig

### Exercises
1. Experiment with a different function (e.g., a cubic plus oscillation) and compare how many extrema you find.
2. Replace the midpoint refinement with a Newton-style update using the second derivative and observe the convergence.
3. Add a tolerance to stop refining when the derivative is close enough to zero, then report the final `h`.

## Approximate π

The integral \(\int_0^1 rac{4}{1+x^2} dx\) equals π. We approximate it with increasingly fine partitions and plot the error to see how fast the quadrature converges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from calculus_with_python.integration import integrate_f

def integrand(x):
    return 4.0 / (1.0 + x ** 2)

n_values = [101, 201, 401, 801, 1601]
errors = []
for n in n_values:
    approx = integrate_f(integrand, 0.0, 1.0, n=n, method="simpson")
    errors.append(abs(approx - np.pi))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(n_values, errors, marker="o")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Error in π approximation vs number of points")
ax.set_xlabel("n")
ax.set_ylabel("|estimate - π|")
ax.grid(True, which="both", linestyle=":")
fig.savefig(assets_dir / "mini_project_pi_error.png", dpi=150, bbox_inches="tight")
plt.close(fig)
fig

### Exercises
1. Try the same integral with trapezoid or Riemann and compare how quickly the error decays.
2. Estimate the area using adaptive spacing: double `n` until the difference between successive Simpson estimates is below \(1e-8\).
3. Use the integral representation of √2 (\(\int_0^1 rac{1}{\sqrt{1+x^2}} dx\)) and compare speed/accuracy with the π approximation.

## Method comparison

Pick a smooth function and compare the derivative/integration methods together: observe how the derivative approximations differ from the analytic derivative and how the integration methods accumulate error. Here we use \(f(x)=\exp(-x^2)\) on \([-2, 2]\).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from calculus_with_python.differentiation import difference_quotient
from calculus_with_python.integration import integrate_f

def gaussian(x):
    return np.exp(-x ** 2)

def gaussian_prime(x):
    return -2 * x * np.exp(-x ** 2)

x = np.linspace(-2.0, 2.0, 401)
fvals = gaussian(x)
true_deriv = gaussian_prime(x)
forward = difference_quotient(gaussian, x, method="forward", h=1e-3)
backward = difference_quotient(gaussian, x, method="backward", h=1e-3)
central = difference_quotient(gaussian, x, method="central", h=1e-3)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, true_deriv, label="analytic")
ax.plot(x, forward, label="forward")
ax.plot(x, backward, label="backward")
ax.plot(x, central, label="central")
ax.set_title("Derivative method comparison for exp(-x^2)")
ax.set_xlabel("x")
ax.set_ylabel("f'(x)")
ax.legend()
ax.grid(True)
fig.savefig(assets_dir / "mini_project_derivative_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)

ns = [101, 201, 401, 801]
quad_errors = {"riemann": [], "trapezoid": [], "simpson": []}
true_integral = integrate_f(gaussian, -2.0, 2.0, n=1601, method="simpson")
for n in ns:
    quad_errors["riemann"].append(abs(integrate_f(gaussian, -2.0, 2.0, n=n, method="riemann", riemann_side="midpoint") - true_integral))
    quad_errors["trapezoid"].append(abs(integrate_f(gaussian, -2.0, 2.0, n=n, method="trapezoid") - true_integral))
    quad_errors["simpson"].append(abs(integrate_f(gaussian, -2.0, 2.0, n=n, method="simpson") - true_integral))

fig2, ax2 = plt.subplots(figsize=(7, 4))
for method, errors in quad_errors.items():
    ax2.plot(ns, errors, marker="o", label=method.capitalize())
ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_title("Integration error comparison on exp(-x^2)")
ax2.set_xlabel("n")
ax2.set_ylabel("Absolute error")
ax2.grid(True, which="both", linestyle=":")
ax2.legend()
fig2.savefig(assets_dir / "mini_project_integration_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig2)
fig

### Exercises
1. Add the second derivative (from the previous notebook) into the comparison plot to show curvature errors.
2. Use the same function but restrict to a narrower interval (e.g., [-1, 1]) and compare how the errors change.
3. Implement Simpson’s rule manually (without `integrate_f`) and verify it matches the helper’s output.